# Host the Qwen3.5 Socratic Tutor on Colab or Kaggle

Run this notebook to get a **temporary HTTPS API URL and a newly generated API key** for
[IITM-2026T3-DSAI-GROUP2/qwen3.5-4b-socratic-tutor](https://huggingface.co/IITM-2026T3-DSAI-GROUP2/qwen3.5-4b-socratic-tutor).

The repository is public and contains the final **LoRA adapter**, rather than a merged model.
This notebook downloads the matching Qwen base, tokenizer, and your adapter automatically.
**No training outputs, datasets, manual adapter paths, HF login, or Cloudflare account are required.**
It uses the exact base revision from training and the uploaded adapter revision checked on 5 November 2026.

1. **Colab:** upload this `.ipynb`, select **Runtime → Change runtime type → T4 GPU** (or a larger NVIDIA GPU).
2. **Kaggle:** import this `.ipynb`, enable a **GPU accelerator** and **Internet**. No dataset attachments or notebook secrets are needed.
3. Use a fresh **interactive** Python 3.11+ session, then **Run all** (not a Kaggle background saved-version run). Wait for the model download, local generation test, and public connection checks.
4. Copy `API_URL` and `API_KEY` from the hosting cell. Keep the session running while your application uses the API.

The model loads in NF4 4-bit with fp16 compute to target a single 16 GB T4. Downloading weights can take several minutes.
GPU availability, session duration, Internet access, and outbound tunnel access depend on your account and platform.
If packages were already imported before installation, restart the session and run the notebook again.

**Keep the notebook private while hosting.** The generated endpoint key is displayed in its output;
clear all outputs before saving or sharing a copy. This is an endpoint-only key, never an HF/Kaggle/Google account token.
Requests pass through Cloudflare. The URL lasts only while the tunnel and notebook session remain alive;
starting hosting again rotates the key. This is a temporary development host, with one generation at a time.


In [ ]:
import os, subprocess, sys
if sys.version_info < (3, 11):
    raise RuntimeError('Use a fresh Colab/Kaggle Python 3.11+ runtime.')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
# Keep the platform's CUDA-enabled Torch; match the saved training environment.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                       'transformers==5.22.1', 'peft==0.21.2', 'accelerate==1.15.0',
                       'bitsandbytes==0.50.2', 'safetensors'])
print('Dependencies installed. Continue below in this fresh session.')


In [ ]:
from pathlib import Path
import json
MODEL_ID = 'IITM-2026T3-DSAI-GROUP2/qwen3.5-4b-socratic-tutor'
MODEL_REVISION = 'd62bd55508533be6705d9f234a5041b477520a9c'
BASE_MODEL_ID = 'Qwen/Qwen3.5-4B'
BASE_REVISION = '851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a'
MAX_CONTEXT_TOKENS = 2048
MAX_NEW_TOKENS = 96
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

def stop_host():
    process = globals().pop('tunnel_process', None)
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=10)
    server = globals().pop('api_server', None)
    if server is not None:
        server.shutdown()
        server.server_close()
    globals()['api_key'] = None
    print('Hosting stopped. Previously displayed credentials no longer serve this session.')

stop_host()
print('Model:', MODEL_ID)
print('Working directory:', WORK_DIR)


## Load directly from Hugging Face

The following cell fetches the **root final adapter**, not the grid-search checkpoints.
It keeps the training base/tokenizer revision, NF4 double quantization, fp16 compute, and disabled thinking.
There is no local adapter folder to select. The public repositories can be downloaded anonymously;
an `HF_TOKEN` environment variable is optional for higher Hub download limits.
Restart the runtime before repeating the model-loading cell to release the previous model's GPU memory.


In [ ]:
import importlib.metadata, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftConfig, PeftModel
if not torch.cuda.is_available():
    raise RuntimeError('Enable an NVIDIA GPU accelerator and restart the runtime before loading.')
print('GPU:', torch.cuda.get_device_name(0))
adapter_config = PeftConfig.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
if (adapter_config.base_model_name_or_path != BASE_MODEL_ID
        or adapter_config.task_type != 'CAUSAL_LM' or adapter_config.peft_type != 'LORA'
        or adapter_config.r != 16 or adapter_config.lora_alpha != 16):
    raise ValueError('The uploaded adapter does not match the expected final training run.')

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, revision=BASE_REVISION, trust_remote_code=False)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                  bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, revision=BASE_REVISION, dtype=torch.float16,
                                          device_map={'': 0}, quantization_config=quantization,
                                          trust_remote_code=False, use_safetensors=True)
model = PeftModel.from_pretrained(base, MODEL_ID, revision=MODEL_REVISION,
                                 is_trainable=False, use_safetensors=True)
model.eval()
probe = tokenizer.apply_chat_template([{'role':'user','content':'U'}, {'role':'assistant','content':'QQQ'}],
                                      tokenize=False, enable_thinking=False)
end_of_turn = probe[probe.index('QQQ') + 3:].split('\n')[0]
eot_ids = tokenizer.encode(end_of_turn, add_special_tokens=False)
assert end_of_turn == '<|im_end|>' and len(eot_ids) == 1, 'Unexpected tokenizer/end-of-turn format'
configured_eos = model.generation_config.eos_token_id
STOP_IDS = sorted({tokenizer.eos_token_id, eot_ids[0],
                  *(configured_eos if isinstance(configured_eos,list) else [] if configured_eos is None else [configured_eos])} - {None})
configuration = getattr(model.config, 'text_config', model.config)
model_limit = getattr(configuration, 'max_position_embeddings', MAX_CONTEXT_TOKENS)
assert type(model_limit) is int and model_limit > 0
MAX_CONTEXT_TOKENS = min(MAX_CONTEXT_TOKENS, model_limit)
metadata = dict(model=MODEL_ID, model_revision=MODEL_REVISION, task='socratic_response_generation',
                base_model=BASE_MODEL_ID, base_revision=BASE_REVISION, quantization='NF4/double/fp16',
                enable_thinking=False, max_new_tokens=MAX_NEW_TOKENS, context_limit=MAX_CONTEXT_TOKENS,
                stop_ids=STOP_IDS, learner_release_approved=False, misconception_classifier=False,
                versions={name:importlib.metadata.version(name) for name in
                          ['torch','transformers','peft','accelerate','bitsandbytes']})
print(json.dumps(metadata, indent=2))


In [ ]:
import time
TRAINING_SYSTEM_TEMPLATE = 'You are a Socratic maths tutor. Guide the student one step at a time with questions and hints. Never state the final answer.\n\nProblem: {question}\nCorrect final answer (for checking only, never reveal it): {answer}'

def infer(body):
    started = time.perf_counter()
    if body['max_new_tokens'] > MAX_NEW_TOKENS:
        raise ValueError('This host preserves the 96-token development generation limit.')
    if 'messages' in body:
        if body['messages'][-1]['role'] != 'user':
            raise ValueError('Provide conversation history ending in a student/user turn.')
        text = tokenizer.apply_chat_template(body['messages'], tokenize=False,
                                             add_generation_prompt=True, enable_thinking=False)
    else:
        text = body['prompt']  # Already formatted using the exact pinned tokenizer/template.
    inputs = tokenizer(text, return_tensors='pt', add_special_tokens=False, truncation=False)
    count = inputs['input_ids'].shape[-1]
    if count < 1 or count + body['max_new_tokens'] > MAX_CONTEXT_TOKENS:
        raise ValueError('Context limit exceeded; do not silently drop conversation or answer constraints.')
    inputs = {k:v.to(model.device) for k,v in inputs.items()}
    with torch.inference_mode():
        output = model.generate(**inputs,max_new_tokens=body['max_new_tokens'],do_sample=False,
                                pad_token_id=tokenizer.pad_token_id,eos_token_id=STOP_IDS,max_time=60)
    generated = output[0,count:]
    response = tokenizer.decode(generated,skip_special_tokens=True).strip()
    # Fail visibly instead of concealing reasoning or extra-turn generation from evaluation.
    protocol_flags = []
    if '<think>' in response or '</think>' in response:
        protocol_flags.append('reasoning_markup_present')
    import re
    if re.search(r'\n\s*(?:user|assistant|model|system)\s*\n',response,re.I):
        protocol_flags.append('possible_invented_next_turn')
    return dict(text=response, model=metadata['model'], task=metadata['task'], input_tokens=count,
                generated_tokens=len(generated), seconds=time.perf_counter()-started,
                output_limit_reached=len(generated)>=body['max_new_tokens'],protocol_flags=protocol_flags,
                learner_release_approved=False, misconception_classifier=False)


In [ ]:
"""Small authenticated, non-streaming inference endpoint for a temporary notebook."""
import json
import secrets
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer


def validate_request(body):
    if not isinstance(body, dict) or set(body) - {"prompt", "messages", "max_new_tokens"}:
        raise ValueError("Use prompt OR messages, plus optional max_new_tokens")
    if ("prompt" in body) == ("messages" in body):
        raise ValueError("Provide exactly one of prompt or messages")
    maximum = body.get("max_new_tokens", 96)
    if type(maximum) is not int or not 1 <= maximum <= 256:
        raise ValueError("max_new_tokens must be an integer from 1 to 256")
    if "prompt" in body:
        if not isinstance(body["prompt"], str) or not body["prompt"].strip() or len(body["prompt"]) > 16000:
            raise ValueError("prompt must contain 1–16000 characters")
    else:
        messages = body["messages"]
        if not isinstance(messages, list) or not 1 <= len(messages) <= 32:
            raise ValueError("messages must contain 1–32 entries")
        for message in messages:
            if (not isinstance(message, dict) or set(message) != {"role", "content"}
                    or message["role"] not in ("system", "user", "assistant")
                    or not isinstance(message["content"], str) or not message["content"].strip()):
                raise ValueError("Invalid chat message")
        if sum(len(m["content"]) for m in messages) > 16000:
            raise ValueError("Chat exceeds 16000 characters")
    return {**body, "max_new_tokens": maximum}


def start_server(infer, metadata, api_key, port=8000):
    if not isinstance(api_key, str) or len(api_key) < 32 or not api_key.isascii() or not api_key.isprintable():
        raise ValueError("Use a random ASCII API key of at least 32 characters")
    busy = threading.Lock()

    class Handler(BaseHTTPRequestHandler):
        def log_message(self, *args):
            pass  # Keep prompts and authentication out of notebook/access logs.

        def send_json(self, status, value):
            payload = json.dumps(value, ensure_ascii=False, allow_nan=False).encode("utf-8")
            self.send_response(status)
            self.send_header("Content-Type", "application/json; charset=utf-8")
            self.send_header("Content-Length", str(len(payload)))
            self.send_header("Cache-Control", "no-store")
            self.end_headers()
            try:
                self.wfile.write(payload)
            except (BrokenPipeError, ConnectionResetError):
                pass

        def authorized(self):
            supplied = self.headers.get("Authorization", "").encode("utf-8")
            if not secrets.compare_digest(supplied, ("Bearer " + api_key).encode("utf-8")):
                self.send_json(401, {"error": "Unauthorized"})
                return False
            return True

        def do_GET(self):
            if not self.authorized():
                return
            if self.path == "/health":
                self.send_json(200, {"status": "ok"})
            elif self.path == "/metadata":
                self.send_json(200, metadata)
            else:
                self.send_json(404, {"error": "Not found"})

        def do_POST(self):
            self.connection.settimeout(15)
            if not self.authorized():
                return
            if self.path != "/predict":
                self.send_json(404, {"error": "Not found"})
                return
            if self.headers.get_content_type() != "application/json" or self.headers.get("Transfer-Encoding"):
                self.send_json(415, {"error": "Send a JSON body with Content-Length"})
                return
            try:
                length = int(self.headers.get("Content-Length", "0"))
                if not 0 < length <= 65536:
                    self.send_json(413, {"error": "JSON body must be 1–65536 bytes"})
                    return
                payload = self.rfile.read(length)
                if len(payload) != length:
                    raise ValueError("Incomplete JSON body")
                body = validate_request(json.loads(payload))
            except (ValueError, UnicodeError, OSError):
                self.send_json(400, {"error": "Invalid request schema or body"})
                return
            if not busy.acquire(blocking=False):
                self.send_json(429, {"error": "Model busy; retry later"})
                return
            try:
                result = infer(body)
                self.send_json(200, result)
            except ValueError:
                self.send_json(422, {"error": "Unsupported input format or model limit exceeded; nothing truncated"})
            except Exception as exc:
                print("Inference failed:", type(exc).__name__, flush=True)
                self.send_json(500, {"error": "Inference failed; check model configuration in notebook"})
            finally:
                busy.release()

    server = ThreadingHTTPServer(("127.0.0.1", port), Handler)
    thread = threading.Thread(target=server.serve_forever, daemon=True)
    thread.start()
    return server


## Check the model and start the authenticated API

This performs one real local generation before exposing the service. A new random key is created automatically.
Every route requires `Authorization: Bearer <API_KEY>`, including `/health` and `/metadata`.
Overlapping generations receive HTTP 429; invalid or oversized requests are rejected.


In [ ]:
import secrets, urllib.request, urllib.error
stop_host()
api_key = secrets.token_urlsafe(32)
api_server = start_server(infer, metadata, api_key, port=8000)
http = urllib.request.build_opener(urllib.request.ProxyHandler({}))
SMOKE_MESSAGES = [
    {'role': 'system', 'content': TRAINING_SYSTEM_TEMPLATE.format(question='Solve 2x + 3 = 11.', answer='4')},
    {'role': 'user', 'content': 'I am stuck. What should I do first?'}
]

def local_request(path, body=None, key=None):
    data = json.dumps(body).encode() if body is not None else None
    headers = {'Content-Type': 'application/json'}
    if key is not None:
        headers['Authorization'] = 'Bearer ' + key
    request = urllib.request.Request('http://127.0.0.1:8000' + path, data=data, headers=headers)
    with http.open(request, timeout=180) as response:
        return json.load(response)

try:
    assert local_request('/health', key=api_key) == {'status': 'ok'}
    for path, body, key, expected in [('/health', None, None, 401),
                                      ('/predict', {'messages': SMOKE_MESSAGES}, 'wrong-key', 401),
                                      ('/predict', {}, api_key, 400),
                                      ('/predict', {'prompt': 'check', 'max_new_tokens': 97}, api_key, 422)]:
        try:
            local_request(path, body, key)
        except urllib.error.HTTPError as error:
            assert error.code == expected, (path, error.code)
        else:
            raise AssertionError('An invalid or unauthenticated request succeeded')
    sample = local_request('/predict', {'messages': SMOKE_MESSAGES, 'max_new_tokens': 32}, api_key)
    assert sample['text'], 'The model returned an empty reply; inspect model setup before publishing.'
    print('Local generation and authentication checks passed.')
    print('Example tutor reply:', sample['text'])
except BaseException:
    stop_host()
    raise


## Create your public URL and display the API key

This downloads the official Linux Cloudflare tunnel client and verifies its publisher-provided SHA-256 digest.
It then checks the public URL with and without authentication before displaying your credentials.
If this cell fails, inspect `tunnel.log` in the working directory. Check that Internet and outbound tunnels are allowed;
a platform or network can block tunnels. Fix the issue and rerun the local-check and tunnel cells.


In [ ]:
import hashlib, os, platform, re, subprocess, time, urllib.error
if platform.system() != "Linux" or platform.machine() not in {"x86_64", "amd64"}:
    raise RuntimeError("Use a standard Linux x86-64 Colab/Kaggle GPU runtime.")
if "tunnel_process" in globals() and tunnel_process.poll() is None:
    raise RuntimeError("Already hosting. Run stop_host(), then rerun the local-check and tunnel cells.")
release_request = urllib.request.Request('https://api.github.com/repos/cloudflare/cloudflared/releases/latest',
                                         headers={'User-Agent': 'qwen-socratic-notebook'})
with urllib.request.urlopen(release_request, timeout=30) as response:
    release = json.load(response)
asset = next(a for a in release['assets'] if a['name'] == 'cloudflared-linux-amd64')
expected = asset.get('digest', '')
assert re.fullmatch(r'sha256:[a-f0-9]{64}', expected), 'No publisher asset digest; do not run an unverified binary.'
assert asset['browser_download_url'].startswith('https://github.com/cloudflare/cloudflared/releases/download/')
binary = WORK_DIR / 'cloudflared'
with urllib.request.urlopen(asset['browser_download_url'], timeout=60) as response, binary.open('wb') as output:
    while data := response.read(1024*1024):
        output.write(data)
with binary.open('rb') as stream:
    assert hashlib.file_digest(stream, 'sha256').hexdigest() == expected.split(':')[1], 'Binary hash mismatch'
binary.chmod(0o700)
log_path = WORK_DIR / 'tunnel.log'
with log_path.open('w') as log:
    tunnel_process = subprocess.Popen([str(binary), 'tunnel', '--url', 'http://127.0.0.1:8000',
                                       '--protocol', 'http2', '--no-autoupdate'], stdout=log, stderr=subprocess.STDOUT)
try:
    endpoint = None
    for _ in range(120):
        if tunnel_process.poll() is not None:
            raise RuntimeError('Tunnel exited; inspect tunnel.log (it contains no API key).')
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_path.read_text())
        if match:
            endpoint = match.group(); break
        time.sleep(.5)
    if endpoint is None:
        raise TimeoutError('No tunnel URL after 60 seconds')
    # The public DNS record and tunnel route may take time to propagate.
    remote_request = urllib.request.Request(endpoint + '/health', headers={'Authorization': 'Bearer ' + api_key})
    deadline = time.monotonic() + 90
    last_error = None
    while time.monotonic() < deadline:
        if tunnel_process.poll() is not None:
            raise RuntimeError('Tunnel exited; inspect tunnel.log (it contains no API key).')
        try:
            with http.open(remote_request, timeout=5) as response:
                if json.load(response).get('status') != 'ok':
                    raise RuntimeError('Public health response was unexpected')
            break
        except urllib.error.HTTPError as error:
            if error.code not in (502, 503, 504):
                raise  # In particular, never ignore a failed authentication check.
            last_error = error
        except (urllib.error.URLError, TimeoutError, OSError) as error:
            last_error = error
        time.sleep(2)
    else:
        raise RuntimeError('Public tunnel did not become reachable; check Internet access and tunnel.log') from last_error
    try:
        http.open(endpoint + '/health', timeout=10)
        raise RuntimeError('Unauthenticated request unexpectedly succeeded')
    except urllib.error.HTTPError as error:
        assert error.code == 401, error.code
    print('API_URL =', endpoint)
    print('API_KEY =', api_key)
    print('PREDICT_URL =', endpoint + '/predict')
    print('cloudflared release:', release['tag_name'], expected)
    print('Keep these credentials private and clear outputs before sharing the notebook. Leave this session running.')
except BaseException:
    stop_host()
    raise


## Use the endpoint from your application

Copy the printed `API_URL` and `API_KEY` into your application's private configuration.
Use the API key in the **Authorization header**, never in a URL. The key is created by this notebook, not by Hugging Face.

| Route | Purpose |
| --- | --- |
| `GET /health` | Authenticated readiness check |
| `GET /metadata` | Repository revisions and inference settings |
| `POST /predict` | Generate a reply; read the response's `text` field |

Example Python client (run in your own application, with `requests` installed):

```python
import requests

API_URL = "https://YOUR-URL.trycloudflare.com"
API_KEY = "YOUR-GENERATED-KEY"
response = requests.post(
    API_URL + "/predict",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={
        "messages": [
            {"role": "system", "content": (
                "You are a Socratic maths tutor. Guide the student one step at a time with questions and hints. "
                "Never state the final answer.\n\nProblem: Solve 2x + 3 = 11.\n"
                "Correct final answer (for checking only, never reveal it): 4"
            )},
            {"role": "user", "content": "I am stuck. What should I do first?"},
        ],
        "max_new_tokens": 96,
    },
    timeout=180,
)
response.raise_for_status()
print(response.json()["text"])
```

Replace the example question and answer with your actual question and reviewed official answer. Build the system
message using `TRAINING_SYSTEM_TEMPLATE` from the inference cell. For a known first attempt, append exactly
`\n\nStudent's first attempt:\n` plus the attempt to that system message. Keep the answer and system message on
your backend. Include the full user/assistant history in each request, ending with a user turn; the server is stateless.
Alternatively send a tokenizer-rendered `prompt` instead of `messages`. The routes use the original notebook's
`/predict` contract; this notebook does not implement the OpenAI API.

Generation is deterministic, thinking is disabled, and the output limit is **96 tokens** (also the default).
Input plus output must fit the **2,048-token serving cap**. Oversized requests fail instead of silently truncating.
Training dialogues were capped at 768 tokens, so longer-context teaching behavior still needs evaluation.
Responses include token counts, timing, output-limit status, and protocol flags; model replies still need
correctness and answer-leak checks in your application. Optional optimized Qwen kernels are omitted;
the portable PyTorch fallback can be slower on a T4.

If you receive HTTP 429, wait for the current generation to finish before retrying.
Closing or idling out the hosted runtime invalidates the URL. Starting the hosting cells again creates a new key.

## Stop hosting

Run `stop_host()` in a new cell, or disconnect/delete the runtime. **Run all leaves hosting running**;
there is deliberately no automatic shutdown cell at the end.
The function is already defined above and can also be used before restarting the hosting cells.

References: [uploaded model](https://huggingface.co/IITM-2026T3-DSAI-GROUP2/qwen3.5-4b-socratic-tutor),
[PEFT model loading](https://huggingface.co/docs/peft/quicktour),
[NF4 loading](https://huggingface.co/docs/transformers/quantization/bitsandbytes),
[Qwen3.5 runtime notes](https://huggingface.co/docs/transformers/model_doc/qwen3_5),
[Cloudflare Quick Tunnels](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/).
